In [1]:
# Задаем период обновления
START_DATE = "2025-07-01"
END_DATE = "2026-07-01"

print("Период:", START_DATE, "—", END_DATE)

Период: 2025-07-01 — 2026-07-01


In [2]:
# Пути к SQL-запросам и CSV-файлам
from pathlib import Path

BASE_PATH = Path.cwd()
SQL_PATH = BASE_PATH / "SQL"
DATA_PATH = BASE_PATH / "Данные"

print("SQL:", SQL_PATH)
print("Данные:", DATA_PATH)

SQL: C:\Users\fmakhmutkhodzhaev\Desktop\КП\КП дашборд\SQL
Данные: C:\Users\fmakhmutkhodzhaev\Desktop\КП\КП дашборд\Данные


In [ ]:
# Автоматизация обновления данных
import csv
import json
import os
import tempfile
import shutil
import pandas as pd

# Файл хранит текущие загруженные периоды для каждой таблицы
PERIODS_FILE = DATA_PATH / "Периоды.json"

# Начальные периоды при первом запуске скрипта
# Roll Rate хранит дополнительный технический месяц перед началом основного периода
INITIAL_PERIODS = {
    "КП": ["2025-07", "2026-06"],
    "Выдачи": ["2025-07", "2026-06"],
    "РППУ": ["2025-07", "2026-06"],
    "Списанные кредиты": ["2025-07", "2026-06"],
    "FPD": ["2025-07", "2026-06"],
    "Винтаж": ["2025-07", "2026-06"],
    "Roll Rate": ["2025-06", "2026-06"]
}

# Загружаем сохраненные периоды или создаем файл при первом запуске
if PERIODS_FILE.exists():
    with open(PERIODS_FILE, "r", encoding="utf-8") as file:
        periods = json.load(file)
else:
    periods = INITIAL_PERIODS.copy()
    with open(PERIODS_FILE, "w", encoding="utf-8") as file:
        json.dump(periods, file, ensure_ascii=False, indent=4)


# Сохраняет актуальные периоды после каждого изменения
def save_periods():
    with open(PERIODS_FILE, "w", encoding="utf-8") as file:
        json.dump(periods, file, ensure_ascii=False, indent=4)


# Выполняет SQL через PostgreSQL COPY и сохраняет результат в CSV
def run_copy(sql, output_file, header=False, append=True):
    sql = sql.strip().rstrip(";")
    connection = engine.raw_connection()
    cursor = connection.cursor()
    try:
        with open(output_file, "ab" if append else "wb") as file:
            cursor.copy_expert(f"COPY ({sql}) TO STDOUT WITH CSV" + (" HEADER" if header else ""), file)
        connection.commit()
    finally:
        cursor.close()
        connection.close()


# Удаляет из CSV месяцы, которые больше не входят в заданный период
def remove_months(csv_file, date_column, start_month, end_month):
    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".csv").name
    try:
        with open(csv_file, "r", encoding="utf-8", newline="") as source, open(temp_file, "w", encoding="utf-8", newline="") as target:
            reader = csv.reader(source)
            writer = csv.writer(target)
            header = next(reader)
            writer.writerow(header)
            date_index = header.index(date_column)

            # Оставляем только строки, попадающие в нужный диапазон месяцев
            for row in reader:
                if start_month <= row[date_index][:7] <= end_month:
                    writer.writerow(row)

        os.replace(temp_file, csv_file)
    finally:
        if os.path.exists(temp_file):
            os.remove(temp_file)


# Обновляет обычные таблицы: КП, Выдачи, РППУ, Списанные кредиты и FPD
def usual_table(name, date_column):
    sql_file = SQL_PATH / f"{name}.sql"
    csv_file = DATA_PATH / f"{name}.csv"
    print(f"\n===== {name} =====")

    current_start = pd.Period(periods[name][0], "M")
    current_end = pd.Period(periods[name][1], "M")

    # Определяем требуемый период из START_DATE и END_DATE
    target_start = pd.Timestamp(START_DATE).to_period("M")
    target_end = (pd.Timestamp(END_DATE) - pd.Timedelta(days=1)).to_period("M")

    # Если период был уменьшен, удаляем лишние месяцы из CSV
    if target_start > current_start or target_end < current_end:
        keep_start = max(target_start, current_start)
        keep_end = min(target_end, current_end)
        remove_months(csv_file, date_column, str(keep_start), str(keep_end))
        periods[name] = [str(keep_start), str(keep_end)]
        save_periods()
        current_start, current_end = keep_start, keep_end

    # Если CSV уже полностью покрывает требуемый период, SQL повторно не выполняем
    if target_start >= current_start and target_end <= current_end:
        print("Актуально")
        return

    with open(sql_file, "r", encoding="utf-8") as file:
        sql_template = file.read()

    # Добавляем недостающие месяцы в конец CSV
    for month in pd.period_range(current_end + 1, target_end, freq="M"):
        sql = sql_template.replace(
            "{START_DATE}",
            month.start_time.strftime("%Y-%m-%d")
        ).replace(
            "{END_DATE}",
            (month.end_time + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
        )

        print(f"Добавляем: {month}")
        run_copy(sql, csv_file, header=False, append=True)
        periods[name][1] = str(month)
        save_periods()

    # Добавляем недостающие месяцы в начало CSV
    for month in reversed(pd.period_range(target_start, current_start - 1, freq="M")):
        sql = sql_template.replace(
            "{START_DATE}",
            month.start_time.strftime("%Y-%m-%d")
        ).replace(
            "{END_DATE}",
            (month.end_time + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
        )

        temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".csv").name

        try:
            # Сначала выгружаем новый месяц во временный файл с заголовком
            run_copy(sql, temp_file, header=True, append=False)

            # Затем добавляем существующий CSV после новых данных
            with open(temp_file, "ab") as target, open(csv_file, "rb") as source:
                shutil.copyfileobj(source, target)

            os.replace(temp_file, csv_file)
            periods[name][0] = str(month)
            save_periods()
        finally:
            if os.path.exists(temp_file):
                os.remove(temp_file)


# Полностью перестраивает Винтаж при изменении начального месяца
# и добавляет только новые месяцы, если изменился конечный месяц
def add_vintage():
    name = "Винтаж"
    csv_file = DATA_PATH / "Винтаж.csv"
    sql_file = SQL_PATH / "Винтаж.sql"

    current_start = pd.Period(periods[name][0], "M")
    current_end = pd.Period(periods[name][1], "M")
    target_start = pd.Timestamp(START_DATE).to_period("M")
    target_end = (pd.Timestamp(END_DATE) - pd.Timedelta(days=1)).to_period("M")

    print("\n===== Винтаж =====")

    with open(sql_file, "r", encoding="utf-8") as file:
        sql_template = file.read()

    # При изменении начального месяца винтаж пересчитывается полностью
    if target_start != current_start:
        temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".csv").name

        try:
            for month in pd.period_range(target_start, target_end, freq="M"):
                sql = sql_template.replace(
                    "{START_DATE}",
                    START_DATE
                ).replace(
                    "{TARGET_MONTH_START}",
                    month.start_time.strftime("%Y-%m-%d")
                ).replace(
                    "{TARGET_MONTH_END}",
                    month.end_time.strftime("%Y-%m-%d")
                )

                run_copy(
                    sql,
                    temp_file,
                    header=(month == target_start),
                    append=(month != target_start)
                )

            os.replace(temp_file, csv_file)
            periods[name] = [str(target_start), str(target_end)]
            save_periods()
        finally:
            if os.path.exists(temp_file):
                os.remove(temp_file)

        return

    # Если конечный месяц уменьшился, удаляем лишние данные
    if target_end < current_end:
        remove_months(
            csv_file,
            "Дата среза",
            str(target_start),
            str(target_end)
        )
        periods[name][1] = str(target_end)
        save_periods()
        current_end = target_end

    # Если появились новые месяцы, добавляем их в конец CSV
    for month in pd.period_range(current_end + 1, target_end, freq="M"):
        sql = sql_template.replace(
            "{START_DATE}",
            START_DATE
        ).replace(
            "{TARGET_MONTH_START}",
            month.start_time.strftime("%Y-%m-%d")
        ).replace(
            "{TARGET_MONTH_END}",
            month.end_time.strftime("%Y-%m-%d")
        )

        print(f"Добавляем: {month}")
        run_copy(sql, csv_file)
        periods[name][1] = str(month)
        save_periods()


# Обновляет Roll Rate по месяцам, сохраняя данные в хронологическом порядке
def add_roll_rate():
    name = "Roll Rate"
    csv_file = DATA_PATH / "Roll Rate.csv"
    sql_file = SQL_PATH / "Roll Rate.sql"

    current_start = pd.Period(periods[name][0], "M")
    current_end = pd.Period(periods[name][1], "M")

    target_start = pd.Timestamp(START_DATE).to_period("M")
    target_end = (pd.Timestamp(END_DATE) - pd.Timedelta(days=1)).to_period("M")

    # Для Roll Rate нужен дополнительный технический месяц
    # перед началом основного анализируемого периода
    roll_rate_start = target_start - 1

    print("\n===== Roll Rate =====")

    # Удаляем месяцы, вышедшие за пределы нового периода
    if roll_rate_start > current_start or target_end < current_end:
        keep_start = max(roll_rate_start, current_start)
        keep_end = min(target_end, current_end)

        remove_months(
            csv_file,
            "Дата среза",
            str(keep_start),
            str(keep_end)
        )

        periods[name] = [str(keep_start), str(keep_end)]
        save_periods()

        current_start, current_end = keep_start, keep_end

    # Если требуемый период уже загружен, ничего не делаем
    if roll_rate_start >= current_start and target_end <= current_end:
        print("Актуально")
        return

    with open(sql_file, "r", encoding="utf-8") as file:
        sql_template = file.read()

    # Добавляем недостающие месяцы в начало CSV в обратном порядке,
    # чтобы итоговые данные оставались хронологически упорядоченными
    for month in reversed(
        pd.period_range(
            roll_rate_start,
            current_start - 1,
            freq="M"
        )
    ):
        month_end = month.end_time.normalize()
        previous_month = month_end.replace(day=1) - pd.DateOffset(months=1)

        sql = sql_template.replace(
            "{PREVIOUS_MONTH}",
            previous_month.strftime("%Y-%m-%d")
        ).replace(
            "{TARGET_MONTH_END}",
            month_end.strftime("%Y-%m-%d")
        )

        temp_file = tempfile.NamedTemporaryFile(
            delete=False,
            suffix=".csv"
        ).name

        try:
            # Сначала выгружаем новый месяц во временный файл с заголовком
            run_copy(
                sql,
                temp_file,
                header=True,
                append=False
            )

            # Затем добавляем существующий CSV после новых данных
            with open(temp_file, "ab") as target, open(csv_file, "rb") as source:
                shutil.copyfileobj(source, target)

            os.replace(temp_file, csv_file)

            periods[name][0] = str(month)
            save_periods()
        finally:
            if os.path.exists(temp_file):
                os.remove(temp_file)

    # Добавляем новые месяцы в конец CSV
    for month in pd.period_range(
        current_end + 1,
        target_end,
        freq="M"
    ):
        month_end = month.end_time.normalize()
        previous_month = month_end.replace(day=1) - pd.DateOffset(months=1)

        sql = sql_template.replace(
            "{PREVIOUS_MONTH}",
            previous_month.strftime("%Y-%m-%d")
        ).replace(
            "{TARGET_MONTH_END}",
            month_end.strftime("%Y-%m-%d")
        )

        print(f"Добавляем: {month}")

        run_copy(
            sql,
            csv_file,
            header=False,
            append=True
        )

        periods[name][1] = str(month)
        save_periods()


# Удаляет дублирующиеся заголовки и приводит заголовки CSV к единому виду
def clean_headers():
    for csv_file in DATA_PATH.glob("*.csv"):
        temp_file = tempfile.NamedTemporaryFile(
            delete=False,
            suffix=".csv"
        ).name

        try:
            with open(
                csv_file,
                "r",
                encoding="utf-8-sig",
                newline=""
            ) as source, open(
                temp_file,
                "w",
                encoding="utf-8",
                newline=""
            ) as target:
                reader = csv.reader(source)
                writer = csv.writer(target)
                header = next(reader)
                writer.writerow(header)

                # Не записываем строки, полностью совпадающие с заголовком
                for row in reader:
                    if row != header:
                        writer.writerow(row)

            os.replace(temp_file, csv_file)
        finally:
            if os.path.exists(temp_file):
                os.remove(temp_file)


# Обновление всех таблиц
usual_table("КП", "Дата среза")
usual_table("Выдачи", "Дата выдачи")
usual_table("РППУ", "Дата РППУ")
usual_table("Списанные кредиты", "Дата списания")
usual_table("FPD", "Дата выдачи")
add_vintage()
add_roll_rate()

# Финальная очистка CSV после всех выгрузок
clean_headers()

print("\n===== ВСЕ ТАБЛИЦЫ ОБНОВЛЕНЫ =====")